# PixiJoy Inpaint (M1) — Colab GPU (quality)

Runtime → **GPU** → **Run all**.

- Auto-fetches **real** training data (no tensorflow_datasets).
- `USE_DRIVE = False` by default (no Google Drive prompt). Set True only to resume after disconnect.
- Pin TFLite only if verify prints **OK**.


In [1]:
!nvidia-smi || true
import tensorflow as tf
print("TF", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices('GPU'), 'Select Runtime → GPU before Run all'


Sun Sep 27 13:13:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
USE_DRIVE = False  # optional resume only

import os, urllib.request, zipfile, shutil
from pathlib import Path
from google.colab import files

ROOT = Path('/content/pixijoy_inpaint')
DRIVE = Path('/content/drive/MyDrive/pixijoy_ml/inpaint')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    (DRIVE / 'dist').mkdir(parents=True, exist_ok=True)

ZIP_URL = 'https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/inpaint_lab.zip'
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL, 'inpaint_lab.zip')
    with zipfile.ZipFile('inpaint_lab.zip') as z:
        z.extractall('.')
    ok = Path('train.py').is_file() and Path('prepare_quality_data.py').is_file()
    print('lab zip OK', ok)
except Exception as e:
    print('zip fetch failed:', e)
if not ok:
    print('Upload inpaint_lab.zip')
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('.')
    ok = Path('train.py').is_file()
assert ok, 'lab zip missing train.py / prepare_quality_data.py — refresh Runtime and re-fetch'
Path('dist').mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    d = Path('dist')
    if d.exists() and not d.is_symlink():
        shutil.rmtree(d) if d.is_dir() else d.unlink()
    if not Path('dist').exists():
        Path('dist').symlink_to(DRIVE / 'dist')


lab zip OK True


In [5]:
print('prepare_quality_data (real data, no TFDS)…')
rc = get_ipython().system('python -u prepare_quality_data.py --data ./data')
print(f'Script finished with exit code: {rc}')
# Treat rc is None or rc == 0 as success
assert (Path('data') / 'READY').is_file() or rc in (0, None), 'prepare_quality_data failed to create READY marker'

prepare_quality_data (real data, no TFDS)…
already ready: 100
Script finished with exit code: None


In [ ]:
from pathlib import Path
import hashlib
USE_DRIVE = globals().get('USE_DRIVE', False)
OUT = Path('/content/drive/MyDrive/pixijoy_ml/inpaint/dist/inpaint.keras') if USE_DRIVE else Path('dist') / 'inpaint.keras'
OUT.parent.mkdir(parents=True, exist_ok=True)
print('ckpt exists' if OUT.with_suffix('.ckpt.keras').is_file() else 'fresh', OUT)

cmd = "python -u train.py --quality --out \"" + str(OUT) + "\""
print('RUN', cmd)
rc = get_ipython().system(cmd)
assert rc in (0, None), 'train failed'

rc = get_ipython().system('python -u export_tflite.py --keras "' + str(OUT) + '" --out "dist/inpaint.tflite"')
assert rc in (0, None), 'export failed (refusing untrained is correct)'

rc = get_ipython().system('python -u verify_contract.py --quality')
assert rc in (0, None), 'verify --quality failed — do NOT pin'

p = Path('dist/inpaint.tflite')
assert p.is_file(), p
print('READY', p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())

fresh dist/inpaint.keras
RUN python -u train.py --quality --out "dist/inpaint.keras"
QUALITY mode: 100 real photos from /content/pixijoy_inpaint/data/rgb
GPUs=[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')] batch=16 epochs=100
out=dist/inpaint.keras
2026-09-27 13:16:25.335511: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1790514985.336650    2679 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:00:05.0, compute capability: 8.0
Model: "pixijoy_inpaint"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (

In [ ]:
from google.colab import files
files.download('dist/inpaint.tflite')
print('Attach inpaint.tflite to ml-packs-v1 ONLY if verify OK')
